# Agent Evaluation with DeepEval

This notebook walks through evaluating the **multi-agent supervisor** — a dice-rolling + statistics agent — using [DeepEval](https://github.com/confident-ai/deepeval).

You will:
1. Define a **golden dataset** of test cases using `Golden` and `EvaluationDataset`
2. Run the agent live and **capture its execution trace**
3. Score the agent with five metrics:
   - `TaskCompletionMetric` — did the agent solve the task?
   - `ToolCorrectnessMetric` — did it call the right sub-agents?
   - `StepEfficiencyMetric` — were the steps efficient?
   - `PlanQualityMetric` — was the plan sound?
   - `GEval` (custom) — does the output contain actual numbers?
4. See **scores, reasons, and pass/fail** rendered inline

## Prerequisites

```bash
uv sync --extra adk --extra eval
```

Copy `.env.example` to `.env` at the project root and fill in `GOOGLE_API_KEY`  
(or set `MODEL_PROVIDER=groq` and `GROQ_API_KEY` for Groq).

In [1]:
import nest_asyncio
nest_asyncio.apply()

import os
import subprocess
from pathlib import Path

_env = Path("../") / ".env"
if _env.exists():
    for _line in _env.read_text().splitlines():
        _line = _line.strip()
        if "=" in _line and not _line.startswith("#"):
            _k, _v = _line.split("=", 1)
            os.environ.setdefault(_k.strip(), _v.strip())
    print(f"Loaded {_env}")
else:
    print("No .env found — expecting API keys already in environment")

print(f"Provider : {os.getenv('MODEL_PROVIDER', 'gemini')}")
print(f"Model: ")

Loaded ../.env
Provider : vertex
Model: 


## Visual helpers

These functions render traces, metric scores, and summary tables as styled HTML — no extra libraries needed.

In [2]:
from IPython.display import HTML, display


def show_dataset(ds):
    """Render an EvaluationDataset as an HTML table."""
    rows = ""
    for i, g in enumerate(ds.goldens):
        tools = " &rarr; ".join(f"<code>{t.name}</code>" for t in (g.expected_tools or []))
        rows += (
            f"<tr style='border-bottom:1px solid #e5e7eb'>"
            f"<td style='padding:10px 12px;color:#9ca3af'>{i}</td>"
            f"<td style='padding:10px 12px'>{g.input}</td>"
            f"<td style='padding:10px 12px'>{tools or '&mdash;'}</td>"
            f"</tr>"
        )
    display(HTML(
        "<table style='border-collapse:collapse;width:100%;font-family:system-ui;font-size:14px'>"
        "<thead><tr style='background:#f9fafb;border-bottom:2px solid #e5e7eb'>"
        "<th style='padding:10px 12px;text-align:left;color:#374151'>#</th>"
        "<th style='padding:10px 12px;text-align:left;color:#374151'>Input</th>"
        "<th style='padding:10px 12px;text-align:left;color:#374151'>Expected tools</th>"
        f"</tr></thead><tbody>{rows}</tbody></table>"
    ))


def show_trace(trace):
    """Render an AgentTrace step-by-step as a colour-coded table."""
    _styles = {
        "User":        ("👤", "#eff6ff", "#1d4ed8"),
        "Tool call":   ("🔧", "#fefce8", "#a16207"),
        "Tool result": ("📥", "#f0fdf4", "#15803d"),
        "Final":       ("✨", "#fdf4ff", "#7e22ce"),
    }
    rows = ""
    for step in trace.steps:
        icon, bg, color = "📋", "#f9fafb", "#374151"
        for prefix, (i, b, c) in _styles.items():
            if step.startswith(prefix):
                icon, bg, color = i, b, c
                break
        rows += (
            f"<tr style='border-bottom:1px solid #f3f4f6'>"
            f"<td style='padding:8px 14px;background:{bg};font-size:18px;width:44px;text-align:center'>{icon}</td>"
            f"<td style='padding:8px 14px;background:{bg};color:{color};"
            f"font-family:monospace;font-size:13px'>{step}</td>"
            f"</tr>"
        )
    display(HTML(
        "<div style='font-family:system-ui;font-size:13px;color:#6b7280;margin:8px 0 4px'>Agent trace</div>"
        f"<table style='border-collapse:collapse;width:100%'>{rows}</table>"
    ))


def show_metric(name, metric, threshold):
    """Render a metric result with score, pass/fail badge, and LLM-judge reasoning."""
    score  = metric.score
    reason = getattr(metric, "reason", "") or ""
    passed = score >= threshold
    color  = "#22c55e" if passed else "#ef4444"
    bg     = "#f0fdf4" if passed else "#fef2f2"
    badge  = "PASS"    if passed else "FAIL"
    emoji  = "✅"      if passed else "❌"
    reason_html = (
        f"<div style='color:#374151;font-size:13px;line-height:1.6;"
        f"border-top:1px solid rgba(0,0,0,0.08);padding-top:8px;margin-top:6px'>"
        f"<em>{reason}</em></div>"
        if reason else ""
    )
    display(HTML(
        f"<div style='border-left:4px solid {color};padding:14px 16px;margin:10px 0;"
        f"background:{bg};border-radius:4px;font-family:system-ui'>"
        f"<div style='display:flex;align-items:center;gap:10px'>"
        f"<span style='font-size:20px'>{emoji}</span>"
        f"<strong style='font-size:15px'>{name}</strong>"
        f"<span style='background:{color};color:white;padding:2px 10px;"
        f"border-radius:12px;font-size:13px;font-weight:700'>{badge}</span>"
        f"<span style='font-size:22px;font-weight:700;color:{color}'>{score:.2f}</span>"
        f"<span style='color:#9ca3af;font-size:13px'>/ threshold&nbsp;{threshold}</span>"
        f"</div>{reason_html}</div>"
    ))


def show_summary(results):
    """Render a summary table for a list of (name, score, threshold) tuples."""
    rows = ""
    for name, score, threshold in results:
        passed      = score >= threshold
        dot_color   = "#22c55e" if passed else "#ef4444"
        badge_bg    = "#dcfce7" if passed else "#fee2e2"
        badge_color = "#15803d" if passed else "#b91c1c"
        rows += (
            f"<tr style='border-bottom:1px solid #f3f4f6'>"
            f"<td style='padding:10px 12px'>"
            f"<span style='display:inline-block;width:10px;height:10px;border-radius:50%;"
            f"background:{dot_color};margin-right:8px'></span>{name}</td>"
            f"<td style='padding:10px 12px;text-align:center'>"
            f"<span style='background:{badge_bg};color:{badge_color};padding:2px 10px;"
            f"border-radius:12px;font-size:13px;font-weight:600'>{'PASS' if passed else 'FAIL'}</span></td>"
            f"<td style='padding:10px 12px;text-align:center;font-weight:700;color:{dot_color}'>{score:.2f}</td>"
            f"<td style='padding:10px 12px;text-align:center;color:#9ca3af'>{threshold}</td>"
            f"</tr>"
        )
    display(HTML(
        "<div style='font-family:system-ui;font-size:13px;color:#6b7280;margin:8px 0 4px'>Evaluation summary</div>"
        "<table style='border-collapse:collapse;width:100%;font-family:system-ui;font-size:14px'>"
        "<thead><tr style='background:#f9fafb;border-bottom:2px solid #e5e7eb'>"
        "<th style='padding:10px 12px;text-align:left;color:#374151'>Metric</th>"
        "<th style='padding:10px 12px;text-align:center;color:#374151'>Result</th>"
        "<th style='padding:10px 12px;text-align:center;color:#374151'>Score</th>"
        "<th style='padding:10px 12px;text-align:center;color:#374151'>Threshold</th>"
        f"</tr></thead><tbody>{rows}</tbody></table>"
    ))


print("Display helpers ready")

Display helpers ready


## The Golden Dataset

A **golden dataset** is a fixed collection of inputs with known-good expectations. In DeepEval, each entry is a `Golden` — it stores:
- `input` — the user query
- `expected_output` — what a correct response must contain
- `expected_tools` — the ordered sub-agent sequence the supervisor should invoke

Wrapping goldens in an `EvaluationDataset` lets you push/pull the dataset from Confident AI:
```python
dataset.push(alias="supervisor-evals")  # store in cloud
dataset.pull(alias="supervisor-evals")  # restore anywhere
```

In [3]:
from tutorials.t07_evaluations.dataset import dataset

print(f"{len(dataset.goldens)} golden cases\n")
show_dataset(dataset)

4 golden cases



#,Input,Expected tools
0,Roll a 6-sided die 5 times and give me the stats,roll_die_agent → stats_agent
1,"Roll a 20-sided die 3 times, then compute mean and standard deviation",roll_die_agent → stats_agent
2,Roll a die once and tell me the result,roll_die_agent
3,Roll a 6-sided die 10 times and give me the statistics,roll_die_agent → stats_agent


## Step 1 — Run the agent and capture the trace

`agent_runner.run_agent()` executes the supervisor via Google ADK's `InMemoryRunner` and returns an `AgentTrace` with:
- **`final_response`** — the text the supervisor emitted at the end
- **`tools_called`** — ordered list of sub-agent names invoked (e.g. `['roll_die_agent', 'stats_agent']`)
- **`steps`** — human-readable record of every action

We'll use the first golden case: *"Roll a 6-sided die 5 times and give me the stats."*

In [4]:
from tutorials.t07_evaluations.agent_runner import run_agent

golden = dataset.goldens[0]
print(f"Input: {golden.input}\n")

trace = run_agent(golden.input)

show_trace(trace)
print(f"\nTools called : {trace.tools_called}")
print(f"\nFinal response:\n{trace.final_response}")

Input: Roll a 6-sided die 5 times and give me the stats



👤,User: Roll a 6-sided die 5 times and give me the stats
🔧,Tool call → roll_die_agent({'request': 'Roll a 6-sided die 5 times'})
📥,"Tool result ← roll_die_agent: {'result': 'Results: 1, 4, 1, 3, 4'}"
🔧,"Tool call → stats_agent({'request': '1, 4, 1, 3, 4'})"
📥,"Tool result ← stats_agent: {'result': 'Mean: 2.6, Std: 1.35646599662505'}"
✨,"Final response: I rolled a 6-sided die 5 times and got the following results: 1, 4, 1, 3, 4. The mean is 2.6 and the standard deviation is 1.36."



Tools called : ['roll_die_agent', 'stats_agent']

Final response:
I rolled a 6-sided die 5 times and got the following results: 1, 4, 1, 3, 4. The mean is 2.6 and the standard deviation is 1.36.


## Step 2 — Task Completion

`TaskCompletionMetric` asks an LLM judge: *"Given the input and the final response, did the agent fully accomplish the task?"*

This is the most important end-to-end metric. An agent can call the right tools and still produce a wrong or incomplete answer — this metric catches that.

In [5]:
from deepeval.metrics import TaskCompletionMetric
from deepeval.test_case import LLMTestCase, ToolCall
from tutorials.t07_evaluations.judge_model import get_judge_model

THRESHOLD_TASK = 0.7

task_test_case = LLMTestCase(
    input=golden.input,
    actual_output=trace.final_response,
    expected_output=golden.expected_output,
    tools_called=[ToolCall(name=t) for t in trace.tools_called],
    expected_tools=golden.expected_tools,
)

task_metric = TaskCompletionMetric(threshold=THRESHOLD_TASK, model=get_judge_model())
task_metric.measure(task_test_case)

show_metric("Task Completion", task_metric, THRESHOLD_TASK)

Output()

## Step 3 — Tool Correctness

`ToolCorrectnessMetric` compares the **actual tool call sequence** against the **expected sequence** from the golden. This is deterministic — no LLM judge needed.

For the supervisor, the correct sequence is:
1. `roll_die_agent` — delegate dice rolling to the sub-agent
2. `stats_agent` — delegate statistics computation

The metric penalises missing tools, extra tools, and wrong ordering.

In [6]:
from deepeval.metrics import ToolCorrectnessMetric

THRESHOLD_TOOLS = 0.8

print(f"Actual   : {trace.tools_called}")
print(f"Expected : {[t.name for t in golden.expected_tools]}")
print()

tool_metric = ToolCorrectnessMetric(threshold=THRESHOLD_TOOLS, model=get_judge_model())
tool_metric.measure(task_test_case)

show_metric("Tool Correctness", tool_metric, THRESHOLD_TOOLS)

Output()

Actual   : ['roll_die_agent', 'stats_agent']
Expected : ['roll_die_agent', 'stats_agent']



## Step 4 — Step Efficiency

A `GEval` judge evaluates whether the agent reached the goal without unnecessary extra steps — repeated tool calls, redundant retries, or wasteful loops.

We pass the **full numbered trace** as `actual_output` so the judge can inspect every step.

> **Note:** DeepEval's built-in `StepEfficiencyMetric` requires its own `@observe`-based tracing and cannot work with a manually-built `LLMTestCase`. `GEval` with a custom criteria string is the equivalent that works with plain text traces.

In [7]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

THRESHOLD_EFFICIENCY = 0.7

trace_text = "\n".join(f"{i+1}. {s}" for i, s in enumerate(trace.steps))

efficiency_test_case = LLMTestCase(
    input=golden.input,
    actual_output=trace_text,
    expected_output=golden.expected_output,
    tools_called=[ToolCall(name=t) for t in trace.tools_called],
    expected_tools=golden.expected_tools,
)

efficiency_metric = GEval(
    name="Step Efficiency",
    criteria=(
        "Evaluate whether the agent reached the goal without unnecessary extra steps. "
        "ACTUAL_OUTPUT is a numbered list of the agent's execution steps. "
        "Score 1.0 if every step was strictly necessary. "
        "Deduct proportionally for repeated tool calls, redundant retries, or wasteful loops. "
        "Score 0.0 if most steps were unnecessary."
    ),
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    threshold=THRESHOLD_EFFICIENCY,
    model=get_judge_model(),
)
efficiency_metric.measure(efficiency_test_case)

show_metric("Step Efficiency", efficiency_metric, THRESHOLD_EFFICIENCY)

Output()

## Step 5 — Plan Quality

A `GEval` judge evaluates the implicit plan the supervisor formed *before* it started executing: was it the right approach? Did it structure the sub-tasks sensibly?

We use a different golden case — 3d20 + stats — to show the metric generalises across inputs.

> **Note:** DeepEval's built-in `PlanQualityMetric` requires its own `@observe`-based tracing and cannot work with a manually-built `LLMTestCase`. `GEval` with a custom criteria string is the equivalent that works with plain text traces.

In [8]:
THRESHOLD_PLAN = 0.7

golden2 = dataset.goldens[1]  # Roll 3d20 + stats
trace2  = run_agent(golden2.input)

show_trace(trace2)
print()

plan_test_case = LLMTestCase(
    input=golden2.input,
    actual_output="\n".join(f"{i+1}. {s}" for i, s in enumerate(trace2.steps)),
    expected_output=golden2.expected_output,
    tools_called=[ToolCall(name=t) for t in trace2.tools_called],
    expected_tools=golden2.expected_tools,
)

plan_metric = GEval(
    name="Plan Quality",
    criteria=(
        "Evaluate whether the agent formed a sound plan to address the user's request. "
        "ACTUAL_OUTPUT is a numbered list of the agent's execution steps. "
        "Score 1.0 if the plan was logical, minimal, and correctly structured. "
        "Deduct for missing steps, wrong ordering, or unnecessary sub-tasks. "
        "Score 0.0 if the plan was fundamentally flawed or absent."
    ),
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    threshold=THRESHOLD_PLAN,
    model=get_judge_model(),
)
plan_metric.measure(plan_test_case)

show_metric("Plan Quality", plan_metric, THRESHOLD_PLAN)

👤,"User: Roll a 20-sided die 3 times, then compute mean and standard deviation"
🔧,Tool call → roll_die_agent({'request': 'Roll a 20-sided die 3 times'})
📥,"Tool result ← roll_die_agent: {'result': 'Results: 8, 8, 10'}"
🔧,"Tool call → stats_agent({'request': 'mean and standard deviation of 8, 8, 10'})"
📥,"Tool result ← stats_agent: {'result': 'Mean: 8.66666666666667, Std: 0.942809041582063'}"
✨,"Final response: I rolled the 20-sided die 3 times and got: 8, 8, 10. The mean is 8.67 and the standard deviation is 0.94."


Output()

## Step 6 — Custom Metric (GEval)

`GEval` is a free-form LLM-as-judge metric where **you write the criteria in plain English**. Use it for domain-specific checks that generic metrics cannot express.

Our custom metric — *Numerical Presence* — checks three concrete requirements:
1. Individual dice roll values appear as explicit numbers
2. A computed mean is labeled and present
3. A computed standard deviation is labeled and present

Each missing component deducts ~0.33 from the score.

In [9]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

THRESHOLD_GEVAL = 0.75

numerical_presence = GEval(
    name="Numerical Presence",
    criteria=(
        "Evaluate whether the output contains ALL THREE of the following: "
        "(1) the individual dice roll results as explicit numbers, "
        "(2) a clearly labeled computed mean value, "
        "(3) a clearly labeled computed standard deviation value. "
        "Score 1.0 if all three are present. "
        "Deduct proportionally for each missing component (0.33 per missing item)."
    ),
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    threshold=THRESHOLD_GEVAL,
    model=get_judge_model(),
)

numerical_presence.measure(task_test_case)

show_metric("Numerical Presence (custom GEval)", numerical_presence, THRESHOLD_GEVAL)

Output()

## Full evaluation suite

Run all five metrics on the same test case and display a summary table — this is the pattern you'd use as a **CI quality gate**.

We use golden case #3 (10d6 + stats) so the full suite runs on a fresh input.

In [10]:
golden3 = dataset.goldens[3]  # Roll 10d6 + stats
trace3  = run_agent(golden3.input)

print(f"Input: {golden3.input}")
show_trace(trace3)

Input: Roll a 6-sided die 10 times and give me the statistics


👤,User: Roll a 6-sided die 10 times and give me the statistics
🔧,Tool call → roll_die_agent({'request': 'Roll 10 dice with 6 sides each'})
📥,"Tool result ← roll_die_agent: {'result': 'Results: 1, 4, 4, 6, 2, 5, 4, 2, 6, 5'}"
🔧,"Tool call → stats_agent({'request': '1, 4, 4, 6, 2, 5, 4, 2, 6, 5'})"
📥,"Tool result ← stats_agent: {'result': 'Mean: 3.9, Std: 1.64012194668567'}"
✨,"Final response: I rolled the dice 10 times and got the following results: 1, 4, 4, 6, 2, 5, 4, 2, 6, 5. The mean is 3.9 and the standard deviation is 1.64."


In [11]:
trace3_text = "\n".join(f"{i+1}. {s}" for i, s in enumerate(trace3.steps))

full_test_case = LLMTestCase(
    input=golden3.input,
    actual_output=trace3_text,
    expected_output=golden3.expected_output,
    tools_called=[ToolCall(name=t) for t in trace3.tools_called],
    expected_tools=golden3.expected_tools,
)

judge = get_judge_model()

suite = [
    ("Task Completion",  TaskCompletionMetric(threshold=0.7, model=judge), 0.7),
    ("Tool Correctness", ToolCorrectnessMetric(threshold=0.8, model=judge), 0.8),
    ("Step Efficiency",  GEval(
        name="Step Efficiency",
        criteria=(
            "Evaluate whether the agent reached the goal without unnecessary extra steps. "
            "ACTUAL_OUTPUT is a numbered list of the agent's execution steps. "
            "Score 1.0 if every step was strictly necessary. "
            "Deduct proportionally for repeated tool calls, redundant retries, or wasteful loops. "
            "Score 0.0 if most steps were unnecessary."
        ),
        evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
        threshold=0.7,
        model=judge,
    ), 0.7),
    ("Plan Quality",     GEval(
        name="Plan Quality",
        criteria=(
            "Evaluate whether the agent formed a sound plan to address the user's request. "
            "ACTUAL_OUTPUT is a numbered list of the agent's execution steps. "
            "Score 1.0 if the plan was logical, minimal, and correctly structured. "
            "Deduct for missing steps, wrong ordering, or unnecessary sub-tasks. "
            "Score 0.0 if the plan was fundamentally flawed or absent."
        ),
        evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
        threshold=0.7,
        model=judge,
    ), 0.7),
    ("Numerical Presence (custom)", GEval(
        name="Numerical Presence",
        criteria=(
            "Evaluate whether the output contains ALL THREE of the following: "
            "(1) individual dice roll results as explicit numbers, "
            "(2) a clearly labeled mean value, "
            "(3) a clearly labeled standard deviation value. "
            "Score 1.0 if all three are present. Deduct 0.33 per missing item."
        ),
        evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
        threshold=0.75,
        model=judge,
    ), 0.75),
]

results = []
for name, metric, threshold in suite:
    print(f"Measuring {name}...", end=" ", flush=True)
    metric.measure(full_test_case)
    results.append((name, metric.score, threshold))
    print(f"{metric.score:.2f}")

print()
show_summary(results)

Measuring Task Completion... 

Output()

1.00
Measuring Tool Correctness... 

Output()

1.00
Measuring Step Efficiency... 

...

Output()

1.00
Measuring Numerical Presence (custom)... 

Output()

1.00



Metric,Result,Score,Threshold
Task Completion,PASS,1.00,0.7
Tool Correctness,PASS,1.00,0.8
Step Efficiency,PASS,1.00,0.7
Plan Quality,PASS,1.00,0.7
Numerical Presence (custom),PASS,1.00,0.75
